In [1]:
from keras.datasets import mnist
from keras.utils import to_categorical

In [2]:
#Showing directories’ size
import os, shutil

train_dir ='./images/train'
validation_dir = './images/val'
test_dir = './images/test'

train_corona_virus_d = './images/train/Corona_Virus_Disease'
train_normal = './images/train/Normal'
train_tuberculosis= './images/train/Tuberculosis'

val_corona_virus_d = './images/val/Corona_Virus_Disease'
val_normal = './images/val/Normal'
val_tuberculosis= './images/val/Tuberculosis'

test_corona_virus_d = './images/test/Corona_Virus_Disease'
test_normal = './images/test/Normal'
test_tuberculosis= './images/test/Tuberculosis'


print('total train corona virus images:', len(os.listdir(train_corona_virus_d)))
print('total train normal images:', len(os.listdir(train_normal)))
print('total train tuberculosis images:', len(os.listdir(train_tuberculosis)))


print('total validation corona virus images:', len(os.listdir(val_corona_virus_d)))
print('total validation normal images:', len(os.listdir(val_normal)))
print('total validation tuberculosis images:', len(os.listdir(val_tuberculosis)))


print('total testing organic corona virus img:', len(os.listdir(test_corona_virus_d)))
print('total testing recicle normal imgs:', len(os.listdir(test_normal)))
print('total testing recicle tuberculosis img:', len(os.listdir(test_tuberculosis)))

total train corona virus images: 1218
total train normal images: 1207
total train tuberculosis images: 1220
total validation corona virus images: 406
total validation normal images: 402
total validation tuberculosis images: 406
total testing organic corona virus img: 407
total testing recicle normal imgs: 404
total testing recicle tuberculosis img: 408


In [3]:
from keras.utils import image_dataset_from_directory

IMG_SIZE = 150
train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode='categorical',
    seed = 100
)

validation_dataset = image_dataset_from_directory(
    validation_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=64,
    label_mode='categorical',
    seed = 100
)

test_dataset = image_dataset_from_directory(
    test_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=64,
    label_mode='categorical',
    seed = 100
)

Found 3645 files belonging to 3 classes.
Found 1214 files belonging to 3 classes.
Found 1219 files belonging to 3 classes.


In [4]:
from keras.applications.vgg16 import VGG16

# Crear el modelo base de VGG16 preentrenado en ImageNet
conv_base = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

In [5]:
# Congelar las capas de la base convolucional para que no se entrenen
conv_base.trainable = False

In [6]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import numpy as np

def objective(trial):
    opt_num_hidden_dense_units = trial.suggest_int("opt_num_hidden_dense_units", 10, 100)
    opt_lr = trial.suggest_float("opt_lr", 1e-6, 1e-2, log=True)
    opt_bs = trial.suggest_int("opt_bs", 16, 128)
    data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2),
    ]
    )
    # Construir el modelo
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = conv_base(x, training=False)
    x = layers.Flatten()(x)
    x = layers.Dense(opt_num_hidden_dense_units, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    # Compilar el modelo
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=opt_lr),
        metrics=['acc'])
    history = model.fit(
        train_dataset,
        epochs=10,
        validation_data=validation_dataset,
        verbose=1,
        batch_size=opt_bs)
    min_val_loss = np.amin(history.history["val_loss"])
    return min_val_loss
    



In [7]:
import optuna as opt
study = opt.create_study()
study.optimize(objective, n_trials=5)


C:\Users\alfre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2024-07-05 09:46:55,765] A new study created in memory with name: no-name-398bcdd5-1a25-41b8-acab-d7f9a5c13753


Epoch 1/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - acc: 0.6322 - loss: 4.1289 - val_acc: 0.8929 - val_loss: 0.3378
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.8203 - loss: 0.5577 - val_acc: 0.9119 - val_loss: 0.2595
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 123s 1s/step - acc: 0.8533 - loss: 0.4133 - val_acc: 0.9226 - val_loss: 0.2340
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - acc: 0.8599 - loss: 0.3872 - val_acc: 0.9292 - val_loss: 0.2000
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.8795 - loss: 0.3340 - val_acc: 0.9448 - val_loss: 0.1689
Epoch 6/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.8991 - loss: 0.2671 - val_acc: 0.9390 - val_loss: 0.1682
Epoch 7/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.8952 - loss: 0.2709 - val_acc: 0.9382 - val_loss: 0.1761
Epoch 8/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.9074 - loss: 0.2767 - val_acc: 0.9563 - val_loss: 0.1385
Epoch 9/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/

[I 2024-07-05 10:07:01,053] Trial 0 finished with value: 0.12515535950660706 and parameters: {'opt_num_hidden_dense_units': 82, 'opt_lr': 0.000158846571192056, 'opt_bs': 68}. Best is trial 0 with value: 0.12515535950660706.


Epoch 1/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - acc: 0.3018 - loss: 18.4779 - val_acc: 0.2397 - val_loss: 18.1269
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.3216 - loss: 15.4931 - val_acc: 0.2718 - val_loss: 13.9822
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.3151 - loss: 12.5827 - val_acc: 0.2982 - val_loss: 11.1291
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.3484 - loss: 10.8299 - val_acc: 0.3353 - val_loss: 8.9546
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.3785 - loss: 9.7509 - val_acc: 0.3690 - val_loss: 7.3029
Epoch 6/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.3987 - loss: 8.1555 - val_acc: 0.4069 - val_loss: 6.1033
Epoch 7/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.4203 - loss: 7.1729 - val_acc: 0.4390 - val_loss: 5.2639
Epoch 8/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.4329 - loss: 6.9050 - val_acc: 0.4646 - val_loss: 4.6556
Epoch 9/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1

[I 2024-07-05 10:27:00,821] Trial 1 finished with value: 3.7058165073394775 and parameters: {'opt_num_hidden_dense_units': 30, 'opt_lr': 1.3735426887826024e-06, 'opt_bs': 111}. Best is trial 0 with value: 0.12515535950660706.


...

[I 2024-07-05 10:47:00,500] Trial 2 finished with value: 0.17974038422107697 and parameters: {'opt_num_hidden_dense_units': 28, 'opt_lr': 0.0020459744216455382, 'opt_bs': 76}. Best is trial 0 with value: 0.12515535950660706.


Epoch 1/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.3802 - loss: 8.5757 - val_acc: 0.6853 - val_loss: 1.9962
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - acc: 0.6214 - loss: 3.0082 - val_acc: 0.7759 - val_loss: 1.1021
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.6841 - loss: 1.9190 - val_acc: 0.8040 - val_loss: 0.7898
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.7214 - loss: 1.4817 - val_acc: 0.8138 - val_loss: 0.6251
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.7480 - loss: 1.1066 - val_acc: 0.8278 - val_loss: 0.5180
Epoch 6/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.7717 - loss: 0.9050 - val_acc: 0.8451 - val_loss: 0.4531
Epoch 7/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.7710 - loss: 0.8031 - val_acc: 0.8542 - val_loss: 0.4099
Epoch 8/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - acc: 0.7759 - loss: 0.7359 - val_acc: 0.8674 - val_loss: 0.3737
Epoch 9/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/

[I 2024-07-05 11:06:52,543] Trial 3 finished with value: 0.3216787278652191 and parameters: {'opt_num_hidden_dense_units': 75, 'opt_lr': 1.8901566221453375e-05, 'opt_bs': 27}. Best is trial 0 with value: 0.12515535950660706.


Epoch 1/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 122s 1s/step - acc: 0.4116 - loss: 8.5354 - val_acc: 0.7265 - val_loss: 1.9605
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.6573 - loss: 3.3921 - val_acc: 0.8138 - val_loss: 1.2289
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.7109 - loss: 2.3586 - val_acc: 0.8418 - val_loss: 0.9315
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.7577 - loss: 1.7619 - val_acc: 0.8624 - val_loss: 0.7571
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.7676 - loss: 1.4904 - val_acc: 0.8756 - val_loss: 0.6141
Epoch 6/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.7860 - loss: 1.1659 - val_acc: 0.8871 - val_loss: 0.5396
Epoch 7/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.7892 - loss: 1.0263 - val_acc: 0.8880 - val_loss: 0.4691
Epoch 8/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - acc: 0.7903 - loss: 0.9415 - val_acc: 0.8937 - val_loss: 0.4219
Epoch 9/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/

[I 2024-07-05 11:26:44,288] Trial 4 finished with value: 0.35341402888298035 and parameters: {'opt_num_hidden_dense_units': 99, 'opt_lr': 1.5506999155079725e-05, 'opt_bs': 43}. Best is trial 0 with value: 0.12515535950660706.


In [8]:
best_params = study.best_params
found_opt_num_hidden_dense_units = best_params["opt_num_hidden_dense_units"]
found_opt_lr = best_params["opt_lr"]
found_opt_bs = best_params["opt_bs"]
print("Found num hidden dense units: {}".format(found_opt_num_hidden_dense_units))
print("Found learning rate: {}".format(found_opt_lr))
print("Found batch size: {}".format(found_opt_bs))

Found num hidden dense units: 82
Found learning rate: 0.000158846571192056
Found batch size: 68


In [ ]:
model.save('./ModelT_Feature_Extraction_WithAugmentation_xray.keras')

In [10]:
from optuna.visualization import plot_contour
plot_contour(study, params=["opt_lr","opt_bs"])

In [11]:
from optuna.visualization import plot_optimization_history
plot_optimization_history(study)